In [1]:
%pwd

'c:\\Users\\HP\\Desktop\\Python\\Vs_Python\\MLoPs\\Movie_recommendation_system\\Notebook'

In [2]:
import os
os.chdir('../')

In [3]:
%pwd

'c:\\Users\\HP\\Desktop\\Python\\Vs_Python\\MLoPs\\Movie_recommendation_system'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass (frozen = True)
class DataValidationConfig:
    root_dir : Path
    unzip_data_dir : Path
    STATUS_FILE : str
    all_schema : dict

In [5]:
from Movie_Recommendation_system.constants import *
from Movie_Recommendation_system.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root]) 

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation

        # Load FULL schema (important for multi-dataset)
        schema = self.schema

        create_directories([Path(config.root_dir)])

        data_validation_config = DataValidationConfig(
            root_dir=Path(config.root_dir),
            unzip_data_dir=Path(config.unzip_dir),
            STATUS_FILE=config.STATUS_FILE,
            all_schema=schema
        )

        return data_validation_config
    

In [7]:
import os
from Movie_Recommendation_system import logger

In [8]:
import pandas as pd

class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self) -> bool:
        try:
            validation_status = True
            schema = self.config.all_schema
            unzip_dir = self.config.unzip_data_dir

            for dataset_name, dataset_schema in schema.items():
                file_path = unzip_dir / dataset_schema["file_name"]

                print(f"\n🔍 Validating dataset: {dataset_name}")
                print(f"📂 File path: {file_path}")

                # 1️⃣ File existence
                if not file_path.exists():
                    print("❌ File does NOT exist")
                    validation_status = False
                    break

                data = pd.read_csv(file_path)
                data_columns = set(data.columns)

                expected_columns = set(dataset_schema["columns"].keys())

                print("Expected columns:", expected_columns)
                print("Actual columns:", data_columns)

                missing_cols = expected_columns - data_columns
                if missing_cols:
                    print("❌ Missing columns:", missing_cols)
                    validation_status = False
                    break
                else:
                    print("✅ All columns matched")

            with open(self.config.STATUS_FILE, "w") as f:
                f.write(f"Validation status: {validation_status}")

            return validation_status

        except Exception as e:
            raise e


In [9]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()

    data_validation = DataValidation(data_validation_config)
    validation_status = data_validation.validate_all_columns()

    if not validation_status:
        raise Exception("Data validation failed")

except Exception as e:
    raise e


[2026-02-10 15:53:37,695: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-02-10 15:53:37,701: INFO: common: yaml file: params.yaml loaded successfully]
[2026-02-10 15:53:37,722: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-02-10 15:53:37,729: INFO: common: created directory at: artifacts]
[2026-02-10 15:53:37,732: INFO: common: created directory at: artifacts\data_validation]

🔍 Validating dataset: movies
📂 File path: artifacts\data_ingestion\UnZip\tmdb_5000_movies.csv
Expected columns: {'status', 'production_countries', 'homepage', 'budget', 'production_companies', 'genres', 'vote_average', 'spoken_languages', 'runtime', 'revenue', 'overview', 'keywords', 'original_title', 'title', 'tagline', 'original_language', 'release_date', 'id', 'popularity', 'vote_count'}
Actual columns: {'status', 'production_countries', 'homepage', 'budget', 'production_companies', 'genres', 'vote_average', 'spoken_languages', 'runtime', 'revenue', 'overview', 'keywor